# OPERA RTC-S1 Quick-Start Demo

This notebook walks through four common tasks with **OPERA RTC-S1**
radar backscatter products using the `rs_tools` package:

1. **Search & download** — find and load Sentinel-1 passes over an AOI
2. **Colour composites** — adjust VV/VH colour limits and use presets
3. **Mosaic / stitch** — combine two passes into a single image
4. **Backscatter conversion** — convert γ⁰ → σ⁰ / β⁰ (fully automated)

## 0. Imports & Area of Interest

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

from rs_tools.config import BoundingBox, SearchConfig
from rs_tools.search import search_archive
from rs_tools.datasets import (
    summarize_search_results,
    print_coverage_report,
    filter_by_coverage,
    records_to_items,
    load_items,
    load_dataset,
)
from rs_tools.datasets.mosaic import mosaic_items
from rs_tools.visualization.rtc_composite import rtc_composite, PRESETS
from rioxarray.merge import merge_arrays

# Antwerp port area
bbox = BoundingBox(west=4.25, south=51.20, east=4.45, north=51.35)

## 1. Search & Download

We search the **Terrascope** archive for one month of data and keep the
two best ascending passes (≥ 80 % spatial coverage).

> **ASF alternative:** replace `"terrascope"` with `"nasa"` in the
> `search_archive` call below (requires a NASA Earthdata account —
> see the [credentials guide](../../docs/credentials.md)).

In [ ]:
config = SearchConfig(
    start_date="2024-06-01",
    end_date="2024-06-30",
    bbox=bbox,
    limit=200,
)

items = search_archive("terrascope", config)
print(f"Found {len(items)} STAC items")

records = summarize_search_results(items, bbox)
print_coverage_report(records)

selected = filter_by_coverage(
    records, min_coverage_pct=80.0, orbit_direction="ascending",
)
print(f"\nSelected {len(selected)} ascending passes with ≥ 80 % coverage")

# Load the first two passes (burst-level items are mosaicked automatically)
pass_items = records_to_items(selected[:2])
data = load_items(pass_items, assets=["VV", "VH"], bbox=bbox, mosaic=True)

for d in data:
    print(f"  {d.label}  shape={d.data['VV'].shape}")

## 2. Colour Composites

Backscatter arrays are converted to amplitude (√power) and mapped to
false-colour RGB:  **R** = VV,  **G** = VH,  **B** = VV.

| Preset | VV range | VH range | Use case |
|--------|----------|----------|----------|
| `default` | 0.129 – 0.871 | 0.040 – 0.358 | Belgium-optimised |
| `OPERA_global` | 0.14 – 0.52 | 0.05 – 0.259 | Original ASF/HyP3 |

In [ ]:
item = data[0]
vv, vh = item.data["VV"].values, item.data["VH"].values

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, preset in zip(axes, ["default", "OPERA_global"]):
    rgb = rtc_composite(vv, vh, preset=preset)
    ax.imshow(rgb, origin="upper")
    ax.set_title(f'Preset: "{preset}"', fontsize=12)
    ax.set_axis_off()
fig.suptitle(f"Colour composite — {item.label}", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
configs = [
    ("Narrow (high contrast)", (0.20, 0.60), (0.06, 0.25)),
    ("Default",                *PRESETS["default"]),
    ("Wide (low saturation)",  (0.05, 1.2),  (0.02, 0.50)),
]
for ax, (label, co, cross) in zip(axes, configs):
    rgb = rtc_composite(vv, vh, co_pol_range=co, cross_pol_range=cross)
    ax.imshow(rgb, origin="upper")
    ax.set_title(f"{label}\nVV={co}, VH={cross}", fontsize=10)
    ax.set_axis_off()
fig.suptitle("Effect of colour limits on contrast", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Mosaic / Stitch Two Passes

Each entry in `data` is already one mosaicked pass (burst strips merged).
Here we combine **two passes** from different dates into a single image
using `merge_arrays`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, d in zip(axes, data[:2]):
    rgb = rtc_composite(d.data["VV"].values, d.data["VH"].values)
    ax.imshow(rgb, origin="upper")
    ax.set_title(d.label, fontsize=11)
    ax.set_axis_off()
fig.suptitle("Individual passes (before stitching)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
vv_stitched = merge_arrays([d.data["VV"] for d in data[:2]])
vh_stitched = merge_arrays([d.data["VH"] for d in data[:2]])

rgb = rtc_composite(vv_stitched.values, vh_stitched.values)

fig, ax = plt.subplots(figsize=(12, 10))
ax.imshow(rgb, origin="upper")
ax.set_title("Stitched mosaic — two passes combined", fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 4. Backscatter Conversion

OPERA RTC-S1 products are distributed as **γ⁰** (gamma-naught).  The
static-layer Area Normalisation Factor (ANF) converts between types:

$$\sigma^0 = \gamma^0 \times \text{ANF}_{\sigma}$$
$$\beta^0 = \gamma^0 \times \text{ANF}_{\beta}$$

`load_dataset` resolves ANF layers and applies the conversion
automatically — no manual steps required.

> On Terrascope (VITO), local static-layer files are used when
> available; otherwise layers are fetched from the STAC catalogue.
> With `archive="nasa"`, layers are resolved from NASA's CMR.

In [ ]:
common = dict(
    short_name="OPERA_RTC_S1",
    bbox=bbox,
    start_date="2024-06-01",
    end_date="2024-06-15",
    archive="terrascope",
    limit=20,
)

g0 = load_dataset(**common, backscatter="gamma0")
s0 = load_dataset(**common, backscatter="sigma0")
b0 = load_dataset(**common, backscatter="beta0")

for label, items in [("γ⁰", g0), ("σ⁰", s0), ("β⁰", b0)]:
    m = items[0]
    vv_mean = float(m.data["VV"].mean(skipna=True))
    vh_mean = float(m.data["VH"].mean(skipna=True))
    print(f"  {label}:  VV mean = {vv_mean:.4f},  VH mean = {vh_mean:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for ax, (label, items) in zip(axes, [
    ("γ⁰ (gamma-0)", g0),
    ("σ⁰ (sigma-0)", s0),
    ("β⁰ (beta-0)",  b0),
]):
    m = items[0]
    rgb = rtc_composite(m.data["VV"].values, m.data["VH"].values)
    ax.imshow(rgb, origin="upper")
    ax.set_title(label, fontsize=13)
    ax.set_axis_off()
fig.suptitle("Backscatter types — same pass", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for label, items, colour in [
    ("γ⁰", g0, "steelblue"),
    ("σ⁰", s0, "darkorange"),
    ("β⁰", b0, "seagreen"),
]:
    vals = items[0].data["VV"].values.ravel()
    vals = vals[np.isfinite(vals) & (vals > 0)]
    ax.hist(vals, bins=200, range=(0, 1.0), density=True, alpha=0.5,
            color=colour, label=f"{label} (mean = {vals.mean():.3f})")
ax.set_xlabel("Linear power", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("VV backscatter distribution — γ⁰ vs σ⁰ vs β⁰", fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Summary

| Section | What you learned |
|---------|-----------------|
| **1. Search** | `search_archive` → `filter_by_coverage` → `load_items` |
| **2. Colour** | `rtc_composite` with presets and custom VV/VH ranges |
| **3. Stitch** | `merge_arrays` to combine passes into a single image |
| **4. Backscatter** | `load_dataset(..., backscatter="sigma0")` for automatic ANF conversion |